# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

# Verbindung zur DuckDB (in-memory reicht völlig)
con = duckdb.connect()

# Hilfsfunktion für SQL-Abfragen
def sql(q):
    return con.sql(q).df()

# Projektpfad bestimmen (Notebook liegt in /notebooks)
DATA = Path("../data/raw/brazilian-ecommerce")

In [2]:
# Alle CSV-Dateien in DuckDB registrieren
for f in DATA.glob("*.csv"):
    name = f.stem.replace("olist_", "").replace("_dataset", "")
    
    con.sql(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM read_csv_auto('{f.as_posix()}')
        """
           )
           
# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [3]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    op.payment_type,
    op.payment_installments,
    op.payment_sequential,
    oi.price,
    oi.freight_value,
    oi.product_id,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_payments op 
        ON o.order_id = op.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
    """)

In [4]:
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP,14409,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,146.87,credit_card,2,1,124.99,21.88,a9516a079e37a9c9c36b9b78b10169e8
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP,09790,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,335.48,credit_card,8,1,289.00,46.48,4aa6014eceb682077f9dc4bffebc05b0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP,01151,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,157.73,credit_card,7,1,139.94,17.79,bd07b66896d6f1494f5b86251848ced7
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP,08775,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,173.30,credit_card,1,1,149.94,23.36,a5647c44af977b148e0a3a4751a09e2e
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP,13056,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,252.25,credit_card,8,1,230.00,22.25,9391a573abe00141c56e38d84d7d5b3b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117596,55a52a925384404b4e57b82938bf6062,0fff4016f4008007ba2bb7d63ebae39d,mogi das cruzes,SP,08780,eb6b32a82d2459d0ce7f2089f9eba0f2,delivered,2018-04-29 21:49:07,2018-05-01 03:15:51,11.80,voucher,1,1,99.90,21.78,78efe838c04bbc568be034082200ac20
117597,0e240f6e253b7d7646700ce3b28fe413,6d443b75cff956bb73dce7385033e266,jundiai,SP,13215,a1fa82769a203e30b8faf81cd32e5193,delivered,2017-09-05 09:29:17,2017-09-06 19:43:28,83.59,voucher,1,2,205.00,0.05,24aba57735be13fd785bc04d1a8812e4
117598,0e240f6e253b7d7646700ce3b28fe413,6d443b75cff956bb73dce7385033e266,jundiai,SP,13215,a1fa82769a203e30b8faf81cd32e5193,delivered,2017-09-05 09:29:17,2017-09-06 19:43:28,23.08,credit_card,1,1,205.00,0.05,24aba57735be13fd785bc04d1a8812e4
117599,0e240f6e253b7d7646700ce3b28fe413,6d443b75cff956bb73dce7385033e266,jundiai,SP,13215,a1fa82769a203e30b8faf81cd32e5193,delivered,2017-09-05 09:29:17,2017-09-06 19:43:28,83.59,voucher,1,6,145.90,90.10,4754d89182db4010eabbf20da5fb7191


In [5]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,payment_installments,payment_sequential,price,freight_value
count,117601,117586,117601.000000,117601.000000,117601.000000,117601.000000,117601.000000
mean,2017-12-30 16:19:35.878368,2017-12-31 03:47:56.832505,172.686752,2.939482,1.093528,120.824783,20.045990
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.000000,1.000000,0.850000,0.000000
25%,2017-09-11 21:42:39,2017-09-12 11:11:34,60.870000,1.000000,1.000000,39.900000,13.080000
50%,2018-01-18 13:37:27,2018-01-18 20:14:08,108.210000,2.000000,1.000000,74.900000,16.290000
75%,2018-05-03 22:43:17,2018-05-04 11:35:16.750000,189.260000,4.000000,1.000000,134.900000,21.190000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080000,24.000000,29.000000,6735.000000,409.680000
std,NaN,NaN,267.592290,2.774223,0.726692,184.479323,15.861315


#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [6]:
df_rfm_eda.dtypes

customer_id                         object
customer_unique_id                  object
customer_city                       object
customer_state                      object
customer_zip_code_prefix            object
order_id                            object
order_status                        object
order_purchase_timestamp    datetime64[us]
order_approved_at           datetime64[us]
payment_value                      float64
payment_type                        object
payment_installments                 int64
payment_sequential                   int64
price                              float64
freight_value                      float64
product_id                          object
dtype: object

In [7]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'payment_value': 'float32', 
              'payment_type': 'category', 
              'payment_installments': 'int16',
              'payment_sequential': 'int16',
              'price': 'float32',
              'freight_value': 'float32',
              'product_id': 'category', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

customer_id                      category
customer_unique_id               category
customer_city                    category
customer_state                   category
customer_zip_code_prefix         category
order_id                         category
order_status                     category
order_purchase_timestamp    datetime64[s]
order_approved_at           datetime64[s]
payment_value                     float32
payment_type                     category
payment_installments                int16
payment_sequential                  int16
price                             float32
freight_value                     float32
product_id                       category
dtype: object

In [8]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,payment_installments,payment_sequential,price,freight_value
count,117601,117586,117601.000000,117601.000000,117601.000000,117601.000000,117601.000000
mean,2017-12-30 16:19:35,2017-12-31 03:47:56,172.686768,2.939482,1.093528,120.824783,20.045990
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.000000,1.000000,0.850000,0.000000
25%,2017-09-11 21:42:39,2017-09-12 11:11:34,60.869999,1.000000,1.000000,39.900002,13.080000
50%,2018-01-18 13:37:27,2018-01-18 20:14:08,108.209999,2.000000,1.000000,74.900002,16.290001
75%,2018-05-03 22:43:17,2018-05-04 11:35:16,189.259995,4.000000,1.000000,134.899994,21.190001
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080078,24.000000,29.000000,6735.000000,409.679993
std,NaN,NaN,267.592285,2.774223,0.726692,184.479324,15.861315


In [9]:
df_rfm_eda.isna().sum()

customer_id                  0
customer_unique_id           0
customer_city                0
customer_state               0
customer_zip_code_prefix     0
order_id                     0
order_status                 0
order_purchase_timestamp     0
order_approved_at           15
payment_value                0
payment_type                 0
payment_installments         0
payment_sequential           0
price                        0
freight_value                0
product_id                   0
dtype: int64

In [10]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id
10500,0bf35cac6cc7327065da879e2d90fae8,c4c0011e639bdbcf26059ddc38bd3c18,varzea paulista,SP,13225,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,39.950001,boleto,1,1,28.990000,10.960000,02a79d79e818ad0be36cfc843a6af7ad
16123,1e101e0daffaddce8159d25a8e53f2b2,c8822fce1d0bfa7ddf0da24fff947172,macae,RJ,27945,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,95.760002,boleto,1,1,79.989998,15.770000,c6dd917a0be2a704582055949915ab32
24681,d5de688c321096d15508faae67a27051,d49f3dae6bad25d05160fc17aca5942d,conselheiro lafaiete,MG,36400,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,60.419998,boleto,1,1,45.900002,14.520000,c3b271f47e73d0c9ccf1b43b7606c705
26892,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,77.059998,boleto,1,1,59.900002,17.160000,7868a64aa111bbb4f41f8e1146c0becb
32364,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,163.429993,boleto,1,1,149.800003,13.630000,cae2e38942c8489d9d7a87a3f525c06b
46467,d85919cb3c0529589c6fa617f5f43281,c094ac95fcd52f821809ec232a7a6956,sao vendelino,RS,95795,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,157.190002,boleto,1,1,133.990005,23.200001,db8ed3d08891d16a2438a67ab3acb740
49044,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,106.809998,boleto,1,1,79.989998,26.820000,c6dd917a0be2a704582055949915ab32
51939,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,54.509998,boleto,1,1,39.990002,14.520000,5ab02ca028398131a5ae91401eb49788
62242,a3d3c38e58b9d2dfb9207cab690b6310,5a4fa4919cbf2b049e72be460a380e5b,abaete,MG,35620,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,154.229996,boleto,1,1,135.000000,19.230000,4fd676d9c4723d475026e40aeae56957
70866,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,349.010010,boleto,1,1,309.899994,39.110001,0e20a07ca1714df21f9b07ca3bf7c682


In [11]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

order_approved_at,False,True
order_status,,
approved,3,0
canceled,566,0
delivered,115020,15
invoiced,371,0
processing,375,0
shipped,1244,0
unavailable,7,0


In [50]:
df_rfm_eda_test = df_rfm_eda
df_rfm_eda_test['order_approved_at'] = df_rfm_eda_test['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda_test['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda_test[df_rfm_eda_test.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id


In [56]:
print(df_rfm_eda_test.shape)
df_rfm_eda_test = df_rfm_eda_test.drop_duplicates()
print('Zeilen nach dem Entfernen von Duplikaten:')
df_rfm_eda_test.shape

(117028, 16)
Zeilen nach dem Entfernen von Duplikaten:


(106528, 16)

In [12]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

Gesamte Duplikate: 10577


In [13]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,0,1,0,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,0,1,0,0,0,0
000229ec398224ef6ca0657da4fc703e,0,0,1,0,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,0,1,0,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...
fffc94f6ce00a00581880bf54a75a037,0,0,1,0,0,0,0
fffcd46ef2263f404302a634eb57f7eb,0,0,1,0,0,0,0
fffce4705a9662cd70adb13d4a31832d,0,0,1,0,0,0,0


In [14]:
test['sum_status']=test.sum(axis=1)

In [15]:
test.loc[test['sum_status']!=1, :] 

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
0008288aa423d2a3f00fcb17cd7d8719,0,0,2,0,0,0,0,2
00143d0f86d6fbd9f9b38ab440ac16f5,0,0,3,0,0,0,0,3
0016dfedd97fc2950e388d2971d718c7,0,0,2,0,0,0,0,2
001ab0a7578dd66cd4b0a71f5b6e1e41,0,0,3,0,0,0,0,3
001d8f0e34a38c37f7dba2a37d4eba8b,0,0,2,0,0,0,0,2
...,...,...,...,...,...,...,...,...
ffd84ab39cd5e873d8dba24342e65c01,0,0,2,0,0,0,0,2
ffe4b41e99d39f0b837a239110260530,0,0,2,0,0,0,0,2
ffecd5a79a0084f6a592288c67e3c298,0,0,3,0,0,0,0,3


####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [16]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [17]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
895ab968e7bb0d5659d16cd74cd1650c,0,0,63,0,0,0,0,63
fedcd9f7ccdc8cba3a18defedd1a5547,0,0,38,0,0,0,0,38
fa65dad1b0e818e3ccc5cb0e39231352,0,0,0,0,0,29,0,29
ccf804e764ed5650cd8759557269dc13,0,0,26,0,0,0,0,26
68986e4324f6a21481df4e6e89abcf01,0,0,24,0,0,0,0,24
...,...,...,...,...,...,...,...,...
607a624a0fb32899edc4e3f9c9b24189,0,0,2,0,0,0,0,2
607d8f6570c73e6a2e553908d572abc7,0,0,2,0,0,0,0,2
60836a0b2a1b5051e61371cf04a16574,0,0,2,0,0,0,0,2


In [18]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id
22112,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,16.700001,voucher,1,17,12.990000,23.209999,ebf9bc6cd600eadd681384e3116fda85
23905,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,1,12.990000,23.209999,ebf9bc6cd600eadd681384e3116fda85
24131,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,16.700001,voucher,1,13,12.990000,23.209999,ebf9bc6cd600eadd681384e3116fda85
24191,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,16.700001,voucher,1,16,12.990000,23.209999,ebf9bc6cd600eadd681384e3116fda85
24221,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,0.240000,voucher,1,19,12.990000,23.209999,ebf9bc6cd600eadd681384e3116fda85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24295,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,16.700001,voucher,1,14,83.800003,5.120000,5ddab10d5e0a23acb99acf56b62b3276
24296,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,9,83.800003,5.120000,5ddab10d5e0a23acb99acf56b62b3276
24297,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,6,12.990000,23.209999,ebf9bc6cd600eadd681384e3116fda85
24298,270c23a11d024a44c896d1894b261a83,9a736b248f67d166d2fbb006bcb877c3,sao paulo,SP,03227,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2.610000,voucher,1,6,12.990000,23.209999,ebf9bc6cd600eadd681384e3116fda85


In [19]:
df_payment = sql("""
SELECT *
FROM order_payments
    """)
test5 = df_payment[df_payment['payment_type'] == 'credit_card']
test5.sort_values(by='payment_sequential', ascending=False).head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
10258,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82
74423,86dda1172108d1601101914c45b51699,2,credit_card,8,300.80
95964,f464a113cdd89833ee98d4a56f41d8c5,2,credit_card,1,20.24
53071,dbbb8658d9a27d2be4cefb9a9a5d9e33,2,credit_card,1,40.14
50388,dabb5a87a6d9cc1388abf76cdcf76e5b,2,credit_card,2,180.00
37443,e003d94282e5f1e45760dde794a8a173,2,credit_card,7,75.79
72388,40edd29fcc7f120710d42d0400bc8e29,2,credit_card,2,150.00
72391,0b7ed5e5b6354118fdeec73b9f9fd8a9,2,credit_card,3,161.54
63497,5df5b6ce88b311095393f226e4366bcf,2,credit_card,8,87.51
76348,df68ea0d1a2a2b9d4dfa236820acbe9c,2,credit_card,8,194.44


In [20]:
test5[test5['order_id'] == 'a079628ac8002126e75f86b0f87332e4']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
10258,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82


In [21]:
df_payment[df_payment['order_id'] == 'a079628ac8002126e75f86b0f87332e4']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
10258,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82
36471,a079628ac8002126e75f86b0f87332e4,2,debit_card,1,50.00


In [22]:
df_payment[df_payment['order_id'] == 'b81ef226f3fe1789b1e8b2acac839d17']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33


In [23]:
testvoucher = df_payment[df_payment['payment_type'] == 'voucher']
testvoucher.sort_values(by='payment_sequential', ascending=False).head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
39108,fa65dad1b0e818e3ccc5cb0e39231352,29,voucher,1,19.26
39111,fa65dad1b0e818e3ccc5cb0e39231352,28,voucher,1,29.05
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
79587,fa65dad1b0e818e3ccc5cb0e39231352,26,voucher,1,28.27
32393,ccf804e764ed5650cd8759557269dc13,26,voucher,1,23.10
39132,ccf804e764ed5650cd8759557269dc13,25,voucher,1,1.53
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
99213,fa65dad1b0e818e3ccc5cb0e39231352,24,voucher,1,0.42
51816,ccf804e764ed5650cd8759557269dc13,24,voucher,1,2.79
60241,ccf804e764ed5650cd8759557269dc13,23,voucher,1,1.03


In [24]:
testvoucher[testvoucher['order_id'] == '895ab968e7bb0d5659d16cd74cd1650c']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
4315,895ab968e7bb0d5659d16cd74cd1650c,17,voucher,1,16.70
9319,895ab968e7bb0d5659d16cd74cd1650c,1,voucher,1,2.61
19993,895ab968e7bb0d5659d16cd74cd1650c,13,voucher,1,16.70
21884,895ab968e7bb0d5659d16cd74cd1650c,16,voucher,1,16.70
29490,895ab968e7bb0d5659d16cd74cd1650c,19,voucher,1,0.24
41528,895ab968e7bb0d5659d16cd74cd1650c,4,voucher,1,2.61
46437,895ab968e7bb0d5659d16cd74cd1650c,21,voucher,1,0.28
52639,895ab968e7bb0d5659d16cd74cd1650c,2,voucher,1,2.61
53054,895ab968e7bb0d5659d16cd74cd1650c,12,voucher,1,16.70
56641,895ab968e7bb0d5659d16cd74cd1650c,18,voucher,1,1.31


In [25]:
order_id = 'a079628ac8002126e75f86b0f87332e4'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)



,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id
27661,62d561a5b1260c4476fd985dcae3eb78,369cf098450e6de108ffa64f68b713a1,capanema,PA,68700,a079628ac8002126e75f86b0f87332e4,delivered,2018-04-24 10:19:45,2018-04-24 18:29:37,102.82,credit_card,3,3,99.989998,52.830002,0983cd4a5cabf1099659ce461511963c
28843,62d561a5b1260c4476fd985dcae3eb78,369cf098450e6de108ffa64f68b713a1,capanema,PA,68700,a079628ac8002126e75f86b0f87332e4,delivered,2018-04-24 10:19:45,2018-04-24 18:29:37,50.00,debit_card,1,2,99.989998,52.830002,0983cd4a5cabf1099659ce461511963c


In [26]:
order_data.groupby('order_id')['payment_value'].nunique().sort_values(ascending=False)

/var/folders/4y/n75k9x_d6nx_z4j3d1jd_z080000gn/T/ipykernel_59262/4152580056.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  order_data.groupby('order_id')['payment_value'].nunique().sort_values(ascending=False)


order_id
a079628ac8002126e75f86b0f87332e4    2
00010242fe8c5a6d1ba2dd792cb16214    0
ab36f41e5d298012acb2179337f1a9db    0
ab3a669dbdc7a8d447bb6fc7a158cf94    0
ab3a6663f21c1a36401300460d3c41cd    0
                                   ..
5561adcb0fd46da4cad3048fa4e7fc00    0
555e60e282181725debc9eb2d69fda3f    0
555e4d40fb6beea866d46eb6a5a01b41    0
555e1afa0cf180760b7ea9f6d8ebc329    0
fffe41c64501cc87c801fd61db3f6244    0
Name: payment_value, Length: 98665, dtype: int64

In [27]:
pd.crosstab(order_data['payment_value'], order_data['order_id'])

order_id,a079628ac8002126e75f86b0f87332e4
payment_value,
50.00,1
102.82,1


In [28]:
normancode = sql("""
SELECT *
FROM order_payments
WHERE order_id = 'a079628ac8002126e75f86b0f87332e4'
ORDER BY payment_sequential;
                                  """)
normancode

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,a079628ac8002126e75f86b0f87332e4,2,debit_card,1,50.00
1,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82


In [29]:
# NaNs nur in order_approved_at
df_rfm_eda[df_rfm_eda['order_approved_at'].isna()].head(10)


,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,payment_type,payment_installments,payment_sequential,price,freight_value,product_id
10500,0bf35cac6cc7327065da879e2d90fae8,c4c0011e639bdbcf26059ddc38bd3c18,varzea paulista,SP,13225,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,39.950001,boleto,1,1,28.990000,10.960000,02a79d79e818ad0be36cfc843a6af7ad
16123,1e101e0daffaddce8159d25a8e53f2b2,c8822fce1d0bfa7ddf0da24fff947172,macae,RJ,27945,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,95.760002,boleto,1,1,79.989998,15.770000,c6dd917a0be2a704582055949915ab32
24681,d5de688c321096d15508faae67a27051,d49f3dae6bad25d05160fc17aca5942d,conselheiro lafaiete,MG,36400,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,60.419998,boleto,1,1,45.900002,14.520000,c3b271f47e73d0c9ccf1b43b7606c705
26892,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,77.059998,boleto,1,1,59.900002,17.160000,7868a64aa111bbb4f41f8e1146c0becb
32364,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,163.429993,boleto,1,1,149.800003,13.630000,cae2e38942c8489d9d7a87a3f525c06b
46467,d85919cb3c0529589c6fa617f5f43281,c094ac95fcd52f821809ec232a7a6956,sao vendelino,RS,95795,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,157.190002,boleto,1,1,133.990005,23.200001,db8ed3d08891d16a2438a67ab3acb740
49044,74bebaf46603f9340e3b50c6b086f992,f79be7c08dd24b72d34634f1b89333a4,sao jose de ribamar,MA,65110,2babbb4b15e6d2dfe95e2de765c97bce,delivered,2017-02-18 17:15:03,NaT,106.809998,boleto,1,1,79.989998,26.820000,c6dd917a0be2a704582055949915ab32
51939,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,54.509998,boleto,1,1,39.990002,14.520000,5ab02ca028398131a5ae91401eb49788
62242,a3d3c38e58b9d2dfb9207cab690b6310,5a4fa4919cbf2b049e72be460a380e5b,abaete,MG,35620,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,154.229996,boleto,1,1,135.000000,19.230000,4fd676d9c4723d475026e40aeae56957
70866,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,349.010010,boleto,1,1,309.899994,39.110001,0e20a07ca1714df21f9b07ca3bf7c682


### EDA für zweite Kernaufgabe

In [32]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    op.payment_value,
    oi.price,
    oi.freight_value,
    p.product_id,
    pcnt.product_category_name_english,
    r.review_score
FROM orders o
JOIN order_payments op ON o.order_id = op.order_id
JOIN order_items oi ON o.order_id = oi.order_id  
JOIN products p ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
    """)

In [33]:
df_pc_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value,product_id,product_category_name_english,review_score
0,73fc7af87114b39712e6da79b0a377eb,delivered,2018-01-11 15:30:49,2018-01-11 15:47:59,397.26,185.00,13.63,fd25ab760bfbba13c198fa3b4f1a0cd3,sports_leisure,4
1,a548910a1c6147796b98fdf73dbeba33,delivered,2018-02-28 12:25:19,2018-02-28 12:48:39,88.09,79.79,8.30,be0dbdc3d67d55727a65d4cd696ca73c,computers_accessories,5
2,f9e4b658b201a9f2ecdecbb34bed034b,delivered,2018-02-03 09:56:22,2018-02-03 10:33:41,194.12,149.00,45.12,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,5
3,658677c97b385a9be170737859d3511b,delivered,2017-04-09 17:41:13,2017-04-09 17:55:19,222.84,179.99,42.85,52c80cedd4e90108bf4fa6a206ef6b03,garden_tools,5
4,8e6bfb81e283fa7e4f11123a3fb894f1,delivered,2018-02-10 10:59:03,2018-02-10 15:48:21,1333.25,1199.00,134.25,3880d25d502b15b1de6fddc42ad1d67a,sports_leisure,5
...,...,...,...,...,...,...,...,...,...,...
116008,b538efecd9ba6d45acff500110ce1e45,delivered,2018-01-24 21:03:02,2018-01-24 21:18:20,33.59,18.49,15.10,8db8c6b5b338e7a8d3332037c2c218f7,telephony,<NA>
116009,9757bb9b0d295ca539b1b15c3eab9b2e,delivered,2017-07-30 20:17:31,2017-07-30 20:30:06,231.27,205.00,26.27,f1c7f353075ce59d8a6f3cf58f419c9c,bed_bath_table,<NA>
116010,8fccc7922426ac1c14efd13857e57bff,delivered,2017-03-19 17:02:45,2017-03-19 17:02:45,271.90,119.90,16.05,e6b6e72a0e6be244a69261788f086429,bed_bath_table,<NA>
116011,8fccc7922426ac1c14efd13857e57bff,delivered,2017-03-19 17:02:45,2017-03-19 17:02:45,271.90,119.90,16.05,e6b6e72a0e6be244a69261788f086429,bed_bath_table,<NA>


In [34]:
df_pc_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value,review_score
count,116013,115999,116013.000000,116013.000000,116013.000000,115066.0
mean,2017-12-31 04:53:31.985683,2017-12-31 16:18:46.135906,172.438748,120.449621,20.062503,4.045913
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.850000,0.000000,1.0
25%,2017-09-12 13:49:33,2017-09-12 21:02:44.500000,61.000000,39.900000,13.080000,4.0
50%,2018-01-18 20:49:36,2018-01-19 10:55:40,108.120000,74.900000,16.320000,5.0
75%,2018-05-04 13:28:13,2018-05-04 19:35:16,189.560000,134.200000,21.220000,5.0
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080000,6735.000000,409.680000,5.0
std,NaN,NaN,266.087022,182.709965,15.834814,1.376048


In [35]:
df_pc_eda.dtypes

order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
payment_value                           float64
price                                   float64
freight_value                           float64
product_id                               object
product_category_name_english            object
review_score                              Int64
dtype: object

In [36]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'payment_value': 'float32', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_id': 'category', 
              'product_category_name_english': 'category',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

order_id                              category
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
payment_value                          float32
price                                  float32
freight_value                          float32
product_id                            category
product_category_name_english         category
review_score                          category
dtype: object

In [37]:
df_pc_eda.describe()

,order_purchase_timestamp,order_approved_at,payment_value,price,freight_value
count,116013,115999,116013.000000,116013.000000,116013.000000
mean,2017-12-31 04:53:31,2017-12-31 16:18:46,172.438751,120.449623,20.062504
min,2016-09-04 21:15:19,2016-10-04 09:43:32,0.000000,0.850000,0.000000
25%,2017-09-12 13:49:33,2017-09-12 21:02:44,61.000000,39.900002,13.080000
50%,2018-01-18 20:49:36,2018-01-19 10:55:40,108.120003,74.900002,16.320000
75%,2018-05-04 13:28:13,2018-05-04 19:35:16,189.559998,134.199997,21.219999
max,2018-09-03 09:06:57,2018-09-03 17:40:06,13664.080078,6735.000000,409.679993
std,NaN,NaN,266.087036,182.709961,15.834814


In [38]:
df_pc_eda.isna().sum()

order_id                           0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
payment_value                      0
price                              0
freight_value                      0
product_id                         0
product_category_name_english      0
review_score                     947
dtype: int64

In [39]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

Gesamte Duplikate: 11410


In [40]:
test2 = pd.crosstab(df_pc_eda['order_status'], df_pc_eda['order_id']).T
test2.head(10)

order_status,approved,delivered,invoiced,processing,shipped
order_id,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,1,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,1,0,0,0
000229ec398224ef6ca0657da4fc703e,0,1,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,1,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,1,0,0,0
00048cc3ae777c65dbb7d2a0634bc1ea,0,1,0,0,0
00054e8431b9d7675808bcb819fb4a32,0,1,0,0,0
000576fe39319847cbb9d288c5617fa6,0,1,0,0,0
0005a1a1728c9d785b8e2b08b904576c,0,1,0,0,0


In [41]:
test2['sum_status']=test2.sum(axis=1)
test2.loc[test2['sum_status']!=1, :] 

order_status,approved,delivered,invoiced,processing,shipped,sum_status
order_id,,,,,,
0008288aa423d2a3f00fcb17cd7d8719,0,2,0,0,0,2
00143d0f86d6fbd9f9b38ab440ac16f5,0,3,0,0,0,3
0016dfedd97fc2950e388d2971d718c7,0,2,0,0,0,2
001ab0a7578dd66cd4b0a71f5b6e1e41,0,3,0,0,0,3
001d8f0e34a38c37f7dba2a37d4eba8b,0,2,0,0,0,2
...,...,...,...,...,...,...
ffd84ab39cd5e873d8dba24342e65c01,0,2,0,0,0,2
ffe4b41e99d39f0b837a239110260530,0,2,0,0,0,2
ffecd5a79a0084f6a592288c67e3c298,0,3,0,0,0,3


In [42]:
test2.loc[test2['sum_status']!=1, :].sort_values('sum_status', ascending=False)

order_status,approved,delivered,invoiced,processing,shipped,sum_status
order_id,,,,,,
895ab968e7bb0d5659d16cd74cd1650c,0,63,0,0,0,63
fedcd9f7ccdc8cba3a18defedd1a5547,0,38,0,0,0,38
fa65dad1b0e818e3ccc5cb0e39231352,0,0,0,0,29,29
ccf804e764ed5650cd8759557269dc13,0,26,0,0,0,26
c6492b842ac190db807c15aff21a7dd6,0,24,0,0,0,24
...,...,...,...,...,...,...
6cdd625fb7db568f0f52136d0a8374a3,0,2,0,0,0,2
6ce409ae9fc35c1c78c3190b660a23b4,0,2,0,0,0,2
6ceabf34d230c31f161988dd2ff8fa92,0,2,0,0,0,2


### EDA für dritte Kernaufgabe

In [43]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
    """)

In [44]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english
0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05,4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,franca,SP,office_furniture
1,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06,5,b8bc237ba3788b23da09c0f1f3a3288c,itajai,SC,sao bernardo do campo,SP,housewares
2,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,sao paulo,SP,office_furniture
3,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10,5,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,mogi das cruzes,SP,office_furniture
4,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15,5,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,campinas,SP,home_confort
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110745,1a9543c90f188e2e4fb14327ad4a9c9b,delivered,2018-01-30 15:28:21,2018-01-31 15:30:30,2018-02-16 16:28:33,2018-03-16 20:03:53,2018-03-13,1,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,curitiba,PR,office_furniture
110746,808c7c69c2778bdf4689eee0286e2bef,canceled,2018-02-22 07:57:07,2018-02-22 08:10:27,NaT,NaT,2018-03-13,1,8a32e327fe2c1b3511609d81aaf9f042,sao paulo,SP,sao paulo,SP,furniture_decor
110747,3304c0c857a9c77a201a551f5a3cacb8,delivered,2017-05-04 12:57:21,2017-05-04 13:10:46,2017-05-05 11:50:12,2017-05-08 10:27:31,2017-05-25,5,ddd51ae8cda92f3995a51fc0f0f3eec7,rio de janeiro,RJ,rio de janeiro,RJ,housewares
110748,cb1f3a44e8b8527e16913306a4d3de2f,delivered,2018-08-07 09:03:02,2018-08-08 09:05:09,2018-08-08 15:01:00,2018-08-15 19:28:29,2018-08-24,4,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,porto alegre,RS,telephony


In [45]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score
count,110750,110736,109605,108457,110750,110750.000000
mean,2018-01-01 12:44:37.691458,2018-01-02 00:18:39.936452,2018-01-05 14:22:58.117586,2018-01-15 00:21:00.601399,2018-01-25 08:59:40.301580,4.035395
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000
25%,2017-09-14 08:05:20,2017-09-14 13:30:22,2017-09-18 22:27:28,2017-09-26 23:18:38,2017-10-05 00:00:00,4.000000
50%,2018-01-20 22:39:33,2018-01-22 13:53:44,2018-01-24 23:32:35,2018-02-03 18:38:41,2018-02-16 00:00:00,5.000000
75%,2018-05-05 15:14:23.750000,2018-05-05 22:13:43,2018-05-08 15:07:00,2018-05-16 12:53:08,2018-05-28 00:00:00,5.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000
std,NaN,NaN,NaN,NaN,NaN,1.385325


In [46]:
df_service_eda.dtypes


order_id                                 object
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_score                              int64
seller_id                                object
seller_city                              object
seller_state                             object
customer_city                            object
customer_state                           object
product_category_name_english            object
dtype: object

In [47]:
df_service_eda.isna().sum()

order_id                            0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  14
order_delivered_carrier_date     1145
order_delivered_customer_date    2293
order_estimated_delivery_date       0
review_score                        0
seller_id                           0
seller_city                         0
seller_state                        0
customer_city                       0
customer_state                      0
product_category_name_english       0
dtype: int64